# Exercise 2: PyTorch core

In this exercise you’ll build core PyTorch “muscle memory” that you’ll reuse in basically every model you write:

- **Autograd**: how gradients are created, how they accumulate, and how to compute gradients for one or multiple inputs.
- **Dataloading**: writing small `Dataset`s, using `DataLoader`, and custom `collate_fn`.
- **Optimizers**: implementing **AdamW** updates from scratch (state, bias correction, weight decay).
- **Training basics**: a clean single training step.
- **Initialization**: fan-in/out and common initializers (Xavier / Kaiming), plus a helper to init `nn.Linear`.

As before: fill in all `TODO`s without changing function names or signatures.
When debugging, print shapes/dtypes/devices, and write tiny sanity checks (e.g. compare to PyTorch’s built-ins).


In [ ]:
from __future__ import annotations
from dataclasses import dataclass
import torch
from torch import nn

## Autograd fundamentals

PyTorch builds a computation graph when you apply operations to tensors with `requires_grad=True`.
Calling `backward()` (or `torch.autograd.grad`) computes gradients by traversing that graph.

### Key concepts
- **Leaf tensor**: a tensor created by you (not the result of an operation) with `requires_grad=True`. Leaf tensors can store gradients in `.grad`.
- **Gradient accumulation**: calling `backward()` adds into `.grad` (it does not overwrite). You must reset gradients between steps/calls.
- **`torch.autograd.grad` vs `.backward()`**
  - `torch.autograd.grad(f, x)` returns `df/dx` directly and does not write into `x.grad` unless you explicitly do so.
  - `f.backward()` writes gradients into `.grad` of leaf tensors.

In the next functions you’ll compute gradients for a simple scalar function such as `f(x) = sum(x^2)` using both APIs.

### `torch.no_grad()`
Wrap inference-only code to avoid tracking gradients and building graphs:
- saves memory
- speeds up evaluation

### `detach()`
`y = x.detach()` returns a tensor that shares data with `x` but is **not connected** to the autograd graph.
This is useful when you want to treat something as a constant target.

### `model.train()` vs `model.eval()`
- `train()` enables training behavior (e.g. dropout active, batchnorm updates running stats).
- `eval()` enables inference behavior (e.g. dropout off, batchnorm uses running stats).

In [ ]:
def grad_with_autograd_grad(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using torch.autograd.grad
    computes the gradients and returns them directly as a standard tensor
    Requirements:
    - Do not call .backward().
    - x should require grad inside the function (don't assume it does).
    - Must return df/dx
    """
    x = x.detach().clone().requires_grad_(True)
    f = (x ** 2).sum()
    (grad_x,) = torch.autograd.grad(f, x)
    return grad_x

autograd_x = torch.tensor([1.0, 2.0, 3.0])
autograd_grad = grad_with_autograd_grad(autograd_x)
print(autograd_grad)

tensor([2., 4., 6.])


In [ ]:

def grad_with_backward(x: torch.Tensor) -> torch.Tensor:
    """
    Compute gradient of f(x) = sum(x^2) using .backward().
    Computes the gradients and stores them inside the .grad attribute of the input tensors
    Requirements:
    - Must return df/dx
    - Must not leak gradients across calls (watch x.grad accumulation)
    """
    #made a fresh tensor to avoid accumulation
    x = x.detach().clone().requires_grad_(True)
    f = (x ** 2).sum()
    f.backward()
    return x.grad

backward_x = torch.tensor([1.0, 2.0, 3.0])
backward_grad = grad_with_backward(backward_x)
print(backward_grad)

In [4]:
def grad_wrt_multiple_inputs(
    a: torch.Tensor, b: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Compute gradients w.r.t. multiple inputs. The function is f(a, b) = sum(a^2 + ab).

    Return:
        (df/da, df/db)

    Requirements:
    - Use torch.autograd.grad
    - Ensure both a and b require grad in this function.
    """
    a = a.detach().clone().requires_grad_(True)
    b = b.detach().clone().requires_grad_(True)
    f = (a ** 2 + a * b).sum()
    grad_a, grad_b = torch.autograd.grad(f, (a, b))
    return grad_a, grad_b

multi_a = torch.tensor([1.0, 2.0, 3.0])
multi_b = torch.tensor([10.0, 20.0, 30.0])
grad_a, grad_b = grad_wrt_multiple_inputs(multi_a, multi_b)
print(grad_a)
print(grad_b)

tensor([12., 24., 36.])
tensor([1., 2., 3.])


## Dataloading

In PyTorch, a `Dataset` defines how to fetch a *single* training example, and a `DataLoader` handles:
- batching
- shuffling
- parallel workers
- optional custom batching logic via `collate_fn`

### `Dataset` in one sentence
A `Dataset` only needs:
- `__len__`: number of items
- `__getitem__`: return one item (e.g. `(x, y)`)

### Why `collate_fn` matters
The default DataLoader collation stacks items along a new batch dimension.
That works for fixed-size tensors, but it breaks for **variable-length sequences**.

So we’ll implement padding ourselves:
- Convert a list of 1D token sequences into a padded tensor `(B, T_max)`
- Track `lengths` and a `padding_mask`

### Mask convention for padding
For padding masks in this exercise:
- `padding_mask[b, t] == True` means **this is padding / invalid**
- `padding_mask[b, t] == False` means **this is a real token**

In [5]:
from torch.utils.data import DataLoader, Dataset

In [7]:
class TensorPairDataset(Dataset):
    """
    Minimal dataset wrapping (x, y).

    x: (N, ...)
    y: (N, ...)

    N is the number of samples. The dataset should return tuples of (x[i], y[i]).
    """

    def __init__(self, x: torch.Tensor, y: torch.Tensor):
        if x.shape[0] != y.shape[0]:
            raise ValueError("x and y must have the same number of samples")
        self.x = x
        self.y = y

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.x[idx], self.y[idx]

pair_x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
pair_y = torch.tensor([0, 1])
pair_dataset = TensorPairDataset(pair_x, pair_y)
print(len(pair_dataset))
print(pair_dataset[1])

2
(tensor([3., 4.]), tensor(1))


In [ ]:
class NextTokenDataset(Dataset):
    """
    Next-token prediction dataset.

    Given tokens of shape (N, T), produce:
      input_ids  = tokens[:, :-1]
      target_ids = tokens[:, 1:]

    Return per item:
      (input_ids, target_ids)

    Notes:
    - Returned tensors should be 1D of length (T-1).
    - dtype should remain integer.
    """

    def __init__(self, tokens: torch.Tensor):
        if tokens.ndim != 2:
            raise ValueError("tokens must have shape (N, T)")
        self.tokens = tokens

    def __len__(self) -> int:
        return self.tokens.shape[0] #returns the number of samples

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.tokens[idx, :-1], self.tokens[idx, 1:]

next_tokens = torch.tensor([[1, 2, 3, 4], [5, 6, 7, 8]])
next_dataset = NextTokenDataset(next_tokens)
print(len(next_dataset))
print(next_dataset[0])

In [ ]:

class RandomCropSequenceDataset(Dataset):
    """
    Sequence dataset that returns random crops of fixed length.

    tokens: (N, T_total)
    crop_len: L

    For each __getitem__:
      - sample a start index s so that s+L <= T_total
      - return tokens[idx, s:s+L]

    Requirements:
    - Use a torch.Generator for deterministic behavior if seed is provided.
    - Do NOT use Python's random module.
    """

    def __init__(self, tokens: torch.Tensor, crop_len: int, seed: int | None = None):
        if tokens.ndim != 2:
            raise ValueError("tokens must have shape (N, T_total)")
        if crop_len <= 0 or crop_len > tokens.shape[1]:
            raise ValueError("crop_len must be in [1, T_total]")
        self.tokens = tokens
        self.crop_len = crop_len
        self.generator = torch.Generator()
        if seed is not None:
            self.generator.manual_seed(seed)

    def __len__(self) -> int:
        return self.tokens.shape[0]

    def __getitem__(self, idx: int) -> torch.Tensor:
        max_start = self.tokens.shape[1] - self.crop_len
        #torch.randint generates a random number from 0 to max_start +1 
        #the generator makes the crop reproducible 
        #.item() makes the tensor an integer
        start = torch.randint(0, max_start + 1, (1,), generator=self.generator).item()
        return self.tokens[idx, start:start + self.crop_len]

crop_tokens = torch.arange(20).reshape(2, 10)
crop_dataset = RandomCropSequenceDataset(crop_tokens, crop_len=4, seed=0)
print(len(crop_dataset))
print(crop_tokens)
print(crop_dataset[0])

2
tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9],
        [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]])
tensor([4, 5, 6, 7])


In [ ]:


@dataclass(frozen=True)
class PaddedBatch:
    """
    A padded batch for variable-length sequences.

    tokens: LongTensor (B, T_max)
    lengths: LongTensor (B,)
    padding_mask: BoolTensor (B, T_max) where True means "this is padding"
    """

    tokens: torch.Tensor
    lengths: torch.Tensor
    padding_mask: torch.Tensor


def pad_1d_sequences(seqs: list[torch.Tensor], pad_value: int = 0) -> PaddedBatch:
    """
    Pad a list of 1D integer tensors to the same length.

    Requirements:
    - Return PaddedBatch(tokens, lengths, padding_mask)
    - padding_mask[b, t] == True iff t >= lengths[b]
    - tokens should be dtype long, if not cast them
    """
    if len(seqs) == 0:
        raise ValueError("seqs must not be empty")
    #new tensors created must be on the same device
    device = seqs[0].device
    #seq.numel() number of elements in the tensor
    #lengths create a 1D tensor with the lengths
    lengths = torch.tensor([seq.numel() for seq in seqs], dtype=torch.long, device=device)
    max_len = int(lengths.max().item()) #find the maximum length
    tokens = torch.full((len(seqs), max_len), pad_value, dtype=torch.long, device=device) #create a tensor with zeros
    #this is faster than extending the tensors

    for i, seq in enumerate(seqs): #overwrites the tensor values in tokens with the input tensors
        tokens[i, : seq.numel()] = seq.to(device=device, dtype=torch.long)

    #creating the padding mask
    positions = torch.arange(max_len, device=device).unsqueeze(0) #one row of 2D tensor 
    padding_mask = positions >= lengths.unsqueeze(1) #column vector of the lengths, checks what is padded
    """ eg:
    Row 1: [0, 1, 2] >= [3]  ->  [0>=3, 1>=3, 2>=3]  ->  [False, False, False]
Row 2: [0, 1, 2] >= [2]  ->  [0>=2, 1>=2, 2>=2]  ->  [False, False, True]"""
    return PaddedBatch(tokens=tokens, lengths=lengths, padding_mask=padding_mask) 

padded = pad_1d_sequences([torch.tensor([1, 2, 3]), torch.tensor([4, 5])], pad_value=0)
print(padded.tokens)
print(padded.lengths)
print(padded.padding_mask)

tensor([[1, 2, 3],
        [4, 5, 0]])
tensor([3, 2])
tensor([[False, False, False],
        [False, False,  True]])


In [ ]:
def collate_next_token_batch(
    batch: list[tuple[torch.Tensor, torch.Tensor]], pad_value: int = 0
) -> dict[str, torch.Tensor]:
    """
    Collate for NextTokenDataset samples that may have variable lengths.
    Used to group the dataset items

    batch: list of (input_ids, target_ids), each 1D

    Return dict with:
      - input_ids: (B, T_max)
      - target_ids: (B, T_max)
      - attention_mask: (B, T_max) where True means "keep" (NOT padding)
      - padding_mask: (B, T_max) where True means "padding"

    Requirements:
    - pad input_ids and target_ids consistently
    - attention_mask is the logical NOT of padding_mask
    """
    input_seqs = [input_ids for input_ids, _ in batch]
    target_seqs = [target_ids for _, target_ids in batch]
    padded_inputs = pad_1d_sequences(input_seqs, pad_value=pad_value)
    padded_targets = pad_1d_sequences(target_seqs, pad_value=pad_value)
    return {
        "input_ids": padded_inputs.tokens,
        "target_ids": padded_targets.tokens,
        "attention_mask": ~padded_inputs.padding_mask,
        "padding_mask": padded_inputs.padding_mask,
    }

collated = collate_next_token_batch([
    (torch.tensor([1, 2, 3]), torch.tensor([2, 3, 4])),
    (torch.tensor([5, 6]), torch.tensor([6, 7])),
])
print(collated)

{'input_ids': tensor([[1, 2, 3],
        [5, 6, 0]]), 'target_ids': tensor([[2, 3, 4],
        [6, 7, 0]]), 'attention_mask': tensor([[ True,  True,  True],
        [ True,  True, False]]), 'padding_mask': tensor([[False, False, False],
        [False, False,  True]])}


In [11]:
def make_dataloader(
    dataset: Dataset,
    batch_size: int,
    shuffle: bool = True,
    drop_last: bool = False,
    collate_fn=None,
    num_workers: int = 0,
) -> DataLoader:
    """
    Create a DataLoader with optional collate_fn.
    """
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        collate_fn=collate_fn,
        num_workers=num_workers,
    )

loader = make_dataloader(pair_dataset, batch_size=2, shuffle=False)
for batch_x, batch_y in loader:
    print(batch_x)
    print(batch_y)
    break

tensor([[1., 2.],
        [3., 4.]])
tensor([0, 1])


## Optimizers (AdamW from scratch)

PyTorch optimizers keep **state** for each parameter (e.g. moment estimates in Adam).
In this section you’ll implement **AdamW**, which is Adam + *decoupled* weight decay.

### AdamW state
For each parameter tensor `p` we store:
- `m`: first moment (EMA of gradients)
- `v`: second moment (EMA of squared gradients)
- `t`: step counter

### Update overview (high level)
1) Update moments `m, v`
2) Bias-correct them (`m_hat, v_hat`)
3) Apply parameter update:
   `p -= lr * ( m_hat / (sqrt(v_hat) + eps) + weight_decay * p )`

Notes:
- This update is **in-place** (mutates `p`).
- Gradients should not be modified.
- State tensors must match parameter shape/device/dtype.

In [12]:
@dataclass
class AdamWState:
    """
    Per-parameter AdamW state.

    m: first moment
    v: second moment
    t: step count
    """

    m: torch.Tensor
    v: torch.Tensor
    t: int


def init_adamw_state(p: torch.Tensor) -> AdamWState:
    """
    Initialize AdamW state tensors for a parameter tensor p.

    What to create:
    - m: zeros like p, same shape/device/dtype
    - v: zeros like p, same shape/device/dtype
    - t: step counter starting at 0

    Notes / requirements:
    - Use torch.zeros_like(p) for m and v.
    - Do NOT attach gradients to the state (initialize under torch.no_grad()).
    - t starts at 0. In adamw_step_, increment t to 1 on the first update *before*
      computing bias correction terms (1 - beta1^t) and (1 - beta2^t).
    - State tensors must live on the same device as p (CPU vs GPU) and have the
      same dtype as p.
    """
    with torch.no_grad():
        return AdamWState(m=torch.zeros_like(p), v=torch.zeros_like(p), t=0)

state_param = torch.tensor([1.0, 2.0])
state = init_adamw_state(state_param)
print(state.m)
print(state.v)
print(state.t)

tensor([0., 0.])
tensor([0., 0.])
0


In [ ]:
def adamw_step_(
    p: torch.Tensor,
    grad: torch.Tensor,
    state: AdamWState,
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> AdamWState:
    """
    In-place AdamW parameter update (updates p).

    Algorithm (AdamW):
      m = beta1*m + (1-beta1)*grad
      v = beta2*v + (1-beta2)*grad^2
      m_hat = m / (1 - beta1^t)
      v_hat = v / (1 - beta2^t)
      p = p - lr * (m_hat / (sqrt(v_hat) + eps) + weight_decay * p)

    Requirements:
    - Update p in-place.
    - Return updated state (with incremented t).
    - Do not modify grad.
    - Should work for any tensor shape.
    """
    beta1, beta2 = betas
    with torch.no_grad():
        state.t += 1
        state.m.mul_(beta1).add_(grad, alpha=1 - beta1)
        state.v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
        """ by using state.m = beta1 * state.m + (1 - beta1) * grad we don't modify the existing tensor
        we create a new tensor and point state.m to that tensor"""

        m_hat = state.m / (1 - beta1 ** state.t)
        v_hat = state.v / (1 - beta2 ** state.t)
        update = m_hat / (torch.sqrt(v_hat) + eps) + weight_decay * p
        p.add_(update, alpha=-lr)
    return state

step_param = torch.tensor([1.0, 2.0])
step_grad = torch.tensor([0.1, -0.2])
step_state = init_adamw_state(step_param)
step_state = adamw_step_(step_param, step_grad, step_state, lr=0.001)
print(step_param)
print(step_state.t)
print(step_state.m)
print(step_state.v)

tensor([0.9990, 2.0010])
1
tensor([ 0.0100, -0.0200])
tensor([1.0000e-05, 4.0000e-05])


In [ ]:
def adamw_step_many_(
    params: list[torch.Tensor],
    grads: list[torch.Tensor],
    states: list[AdamWState],
    lr: float,
    betas: tuple[float, float] = (0.9, 0.999),
    eps: float = 1e-8,
    weight_decay: float = 0.01,
) -> list[AdamWState]:
    """
    Apply AdamW to many parameters.

    Requirements:
    - len(params) == len(grads) == len(states)
    - Update each param in-place.
    - Return the list of updated states.
    """
    if not (len(params) == len(grads) == len(states)):
        raise ValueError("params, grads, and states must have the same length")

    updated_states = []
    for p, grad, state in zip(params, grads, states): #zip groups p = p1, grad = g1, state = s1
        updated_states.append(
            adamw_step_(
                p,
                grad,
                state,
                lr=lr,
                betas=betas,
                eps=eps,
                weight_decay=weight_decay,
            )
        )
    return updated_states

many_params = [torch.tensor([1.0, 2.0]), torch.tensor([[3.0], [4.0]])]
many_grads = [torch.tensor([0.1, 0.2]), torch.tensor([[0.3], [0.4]])]
many_states = [init_adamw_state(p) for p in many_params] #state param --> adamW state
many_states = adamw_step_many_(many_params, many_grads, many_states, lr=0.001)
print(many_params)
print([state.t for state in many_states])

[tensor([0.9990, 1.9990]), tensor([[2.9990],
        [3.9990]])]
[1, 1]


## Training basics

A minimal training step follows the same pattern almost everywhere:

1) set model to train mode
2) reset gradients
3) forward pass
4) compute loss
5) backward pass
6) step optimizer

In this exercise you’ll implement a single MSE training step using a standard PyTorch optimizer.
Return a Python float loss value.

In [ ]:
def train_step_mse(
    model: nn.Module,
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
) -> float:
    """
    One MSE train step using standard torch optimizer.
    """
    model.train()
    x, y = batch
    optimizer.zero_grad() #resetting the old gradients
    pred = model(x)
    loss = torch.mean((pred - y) ** 2)
    loss.backward()  #torch.autograd.grad doesnt store the gradients in .grad
    optimizer.step()
    return loss.item()

train_model = nn.Linear(2, 1) #y = xW^T + b
train_batch = (torch.tensor([[1.0, 2.0], [3.0, 4.0]]), torch.tensor([[1.0], [2.0]]))
train_optimizer = torch.optim.SGD(train_model.parameters(), lr=0.01)
train_loss = train_step_mse(train_model, train_batch, train_optimizer)
print(train_loss)


1.5008893013000488


## Parameter initialization

Initialization matters because it controls signal and gradient scales at the start of training.

### Fan-in / fan-out
- `fan_in`: number of input connections to a unit
- `fan_out`: number of output connections from a unit

For a Linear layer weight of shape `(out_features, in_features)`:
- `fan_in = in_features`
- `fan_out = out_features`

### Common schemes
- **Xavier / Glorot** (often good for tanh / linear-ish nets):
  keeps variance stable across layers when activations are roughly symmetric.
- **Kaiming / He** (often good for ReLU-like nets):
  accounts for the fact that ReLU zeroes out about half the inputs.

In this section you’ll implement Xavier uniform and Kaiming uniform and use them to initialize `nn.Linear`.
We also always zero the bias unless explicitly told otherwise.

In [19]:
def fan_in_fan_out(weight: torch.Tensor) -> tuple[int, int]:
    """Compute (fan_in, fan_out) for a weight tensor."""
    if weight.ndim < 2:
        raise ValueError("weight must have at least 2 dimensions")
    receptive_field_size = 1
    if weight.ndim > 2:
        receptive_field_size = weight[0][0].numel()  #gets the output and input channel
    #.numel() counts how many numbers are in that kernel
    fan_in = weight.shape[1] * receptive_field_size
    fan_out = weight.shape[0] * receptive_field_size
    return fan_in, fan_out

linear_weight = torch.empty(4, 3)
print(fan_in_fan_out(linear_weight))

(3, 4)


In [20]:
def xavier_uniform_(weight: torch.Tensor, gain: float = 1.0) -> torch.Tensor:
    """
    In-place Xavier/Glorot uniform init:
      bound = gain * sqrt(6 / (fan_in + fan_out))
      U(-bound, bound)
    """
    fan_in, fan_out = fan_in_fan_out(weight)
    bound = gain * (6.0 / (fan_in + fan_out)) ** 0.5
    with torch.no_grad():
        weight.uniform_(-bound, bound)
    return weight

xavier_weight = torch.empty(4, 3)
xavier_uniform_(xavier_weight)
print(xavier_weight)
print(xavier_weight.shape)

tensor([[ 0.0933,  0.9171, -0.1762],
        [-0.5854,  0.0916,  0.6647],
        [-0.1930, -0.4127, -0.2170],
        [ 0.8297, -0.6194,  0.0350]])
torch.Size([4, 3])


In [ ]:
def kaiming_uniform_(weight: torch.Tensor, nonlinearity: str = "relu") -> torch.Tensor:
    """
    In-place Kaiming/He uniform init.

    Follow this common choice:
      gain = sqrt(2) for ReLU
      std = gain / sqrt(fan_in)
      bound = sqrt(3) * std
      U(-bound, bound)
    """
    fan_in, _ = fan_in_fan_out(weight)
    gain = 2.0 ** 0.5 if nonlinearity == "relu" else 1.0
    std = gain / (fan_in ** 0.5)
    bound = (3.0 ** 0.5) * std
    with torch.no_grad(): #NO NEED to track the gradient during the initialization process
        weight.uniform_(-bound, bound)
    return weight

kaiming_weight = torch.empty(4, 3)
kaiming_uniform_(kaiming_weight)
print(kaiming_weight)
print(kaiming_weight.shape)

tensor([[ 0.4068, -0.0268,  0.0719],
        [-1.0885, -0.0052,  0.1619],
        [-0.9687, -0.1459, -1.1691],
        [ 1.3002, -1.4140,  1.0201]])
torch.Size([4, 3])


In [22]:
def init_linear_(layer: nn.Linear, scheme: str = "xavier") -> nn.Linear:
    """
    Initialize an nn.Linear in-place.

    scheme:
      - "xavier"
      - "kaiming_relu"
      - "zero" (weights and bias = 0)
    """
    if scheme == "xavier":
        xavier_uniform_(layer.weight)
    elif scheme == "kaiming_relu":
        kaiming_uniform_(layer.weight, nonlinearity="relu")
    elif scheme == "zero":
        with torch.no_grad():
            layer.weight.zero_()
    else:
        raise ValueError(f"Unknown initialization scheme: {scheme}")

    if layer.bias is not None:
        with torch.no_grad():
            layer.bias.zero_()
    return layer

init_layer = nn.Linear(3, 2)
init_linear_(init_layer, scheme="xavier")
print(init_layer.weight)
print(init_layer.bias)

Parameter containing:
tensor([[ 0.0981, -0.6755,  0.1145],
        [ 0.4090, -0.6159,  0.8619]], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)
